In [3]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)

In [4]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

SyntaxError: trailing comma not allowed without surrounding parentheses (2873290629.py, line 3)

In [5]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap_fori
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f


sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 3
degree = 3
n_elem = 100
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)], [(0.2, 10)], [(0.2, 10)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections_indexed=cross_sections)
solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
source_mapped                        = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep                         = jnp.zeros((domain.n_elements, element_type.n_basis))

quad_set = jsn.quadrature_set.create_tn_quadrature_set(4)
toposorts = [TopologicalSortedAngle.from_domain_angle(domain, angle, 100) for angle in quad_set.angles]
total_toposorts = jax.tree.map(lambda * xs : jnp.stack(xs), *toposorts)

In [6]:
source_angle = jax.tree.map(lambda ts : jnp.repeat(ts, total_toposorts.angle.shape[0], axis=0).reshape((total_toposorts.angle.shape[0],)+ts.shape), source_mapped)

In [7]:
total_buffer_sweep = jnp.zeros((total_toposorts.angle.shape[0], domain.n_elements, element_type.n_basis))

In [ ]:
from jax_sn.solution_domain.transport_operator import _setup_elements_idx_in_precomp_with_omega_n_calculation
@jax.jit
def transport_inverse_sweep_complete_vmap_fori(geometric_domain           : GeometricDomain, 
                                               fem_matrices               : FEMMatrices, 
                                               precomputed_geometric_data : PrecomputedGeometricData, 
                                               cross_sections             : CrossSectionDataSingleGroup,                                                
                                               topological_sorted_angle   : TopologicalSortedAngle, 
                                               precomputed_source         : Source, 
                                               sweep_result               : jnp.ndarray) -> jnp.ndarray:    
    '''
    Perform a complete transport inverse sweep over all elements in the geometric domain using vmap and fori_loop.

    vmap is used to compute all Gij_inv, QM_i and Dij_f_p_fm matrices for all elements in a vectorized manner.
    Then, a fori_loop is used to iterate over the elements in topological order to compute the sweep result.
    This takes significant memory due to the large intermediate matrices. 

    Parameters
    ----------
    geometric_domain : GeometricDomain
        The geometric domain data.
    fem_matrices : FEMMatrices
        The finite element method matrices.
    precomputed_geometric_data : PrecomputedGeometricData
        The precomputed geometric data (Jacobian determinants and inverses).
    cross_sections : CrossSectionDataSingleGroup
        The cross section data.
    topological_sorted_angle : TopologicalSortedAngle
        The topologically sorted angle data.
    precomputed_source : Source
        The source data.
    sweep_result : jnp.ndarray [n_elements, n_basis]
        The initial sweep result array (usually zeros).
    Returns
    -------
    sweep_result : jnp.ndarray [n_elements, n_basis]    
        The computed sweep result array after performing the transport sweep.
    '''    
    

    gij_inv_all, qm_i_all, dij_f_p_fm_all, _, abs_omega_dot_n_Je = _setup_elements_idx_in_precomp_with_omega_n_calculation(topological_sorted_angle.levels_argsort, geometric_domain, fem_matrices,
                                                            precomputed_geometric_data, cross_sections, topological_sorted_angle.angle,
                                                            precomputed_source)    
    return gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je



In [22]:

gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je = transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep)

In [95]:



def iota_kernel(topo_sort, o_ref):

  def _body(i, o_ref):
      o_ref[i] = topo_sort[i]
      return o_ref

  jax.lax.fori_loop(0, topo_sort.shape[0], _body, o_ref)


@partial(jax.jit, static_argnums=(0,))
def iota(size: int):
  return pl.pallas_call(iota_kernel,
                        out_shape=jax.ShapeDtypeStruct((size,), jnp.int32),
                        grid=(1,))(toposorts[0].levels_argsort)



In [96]:
solution_domain.domain.n_elements

60000

In [98]:
%timeit iota(solution_domain.domain.n_elements).block_until_ready()

4.39 ms ± 88.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [100]:
iota(solution_domain.domain.n_elements).shape

(60000,)

In [63]:
toposorts[0]

TopologicalSortedAngle(levels=Array([38, 37, 38, ...,  0,  1,  0], dtype=int64), angle=Array([-0.09901475, -0.09901475, -0.99014754], dtype=float64), levels_argsort=Array([   57,   117,   177, ..., 54000, 54002, 54602], dtype=int64))

In [53]:
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
def sweep_pallas(elements_topological_sorting, gij_inv_all, qm_i_all,
                  dij_f_p_fm_all, abs_omega_dot_n_Je, etoe,
                  sweep_result_init):

    # capture all data in a closure
    @jax.jit
    def kernel(o_ref):
        i = 0
        N = elements_topological_sorting.shape[0]

        
        elem_idx = elements_topological_sorting[i]

        # gij_inv    = gij_inv_all[i]
        # qm_i       = qm_i_all[i]
        # dij_matrix = dij_f_p_fm_all[i]
        # omega      = abs_omega_dot_n_Je[i]
        # neighbors  = etoe[elem_idx]

        # values = o_ref[neighbors]

        # # collapsed_dij = jnp.einsum("f, f i j, f j -> i",
        # #                             omega, dij_matrix, values)

        # result = gij_inv @ (qm_i )#+ collapsed_dij)

        o_ref[elem_idx] = result

        #return o_ref
    def ree():
        return pl.pallas_call(
            kernel,
            out_shape=jax.ShapeDtypeStruct(sweep_result_init.shape,
                                       sweep_result_init.dtype),
            grid=(1,),  # scalar grid
        )
    
    return ree()


result = sweep_pallas(
    elements_topological_sorting=toposorts[0].levels_argsort,
    gij_inv_all=gij_inv_all,
    qm_i_all=qm_i_all,
    dij_f_p_fm_all=dij_f_p_fm_all,
    abs_omega_dot_n_Je=abs_omega_dot_n_Je,
    etoe=solution_domain.domain.etoe,
    sweep_result_init=buffer_sweep
)

In [54]:
result()

TypeError: Argument '<PjitFunction of <function _pallas_call.<locals>.wrapped at 0x7ae717bc7740>>' of type '<class 'jaxlib._jax.PjitFunction'>' is not a valid JAX type

In [27]:
%timeit transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep)[3].block_until_ready()

32.7 ms ± 822 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
from jax.experimental import pallas as pl



def 

AttributeError: module 'jax.experimental.pallas' has no attribute 'kernel'

In [10]:
result_bs = transport_inverse_sweep_multiple_angles_batched_single_f(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 70).block_until_ready()

In [11]:
result_bs = transport_inverse_sweep_multiple_angles_batched(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 100).block_until_ready()

In [35]:
topolevels = [jsn.solution_domain.transport_operator_levels.TopologicalSortedAnglePrecomputedLevels.from_topological_sorted_angle(ts) for ts in toposorts]

In [61]:
sizes = [tl.sizes for tl in topolevels]

In [62]:
print(sizes)

[Array([2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048,
       2048, 2048, 1024, 1024, 1024,  512,  256], dtype=int32), Array([ 16,  16,  32,  32,  32,  32,  64,  64,  64,  64,  64,  64, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256

In [65]:
jnp.min(jnp.bincount(topolevels[0].levels))

Array(183, dtype=int64)

In [60]:
for i in topolevels:
    print(jnp.min(i.sizes))

256
16
256
1
1
256
16
256
1
16
16
1
1
16
16
1
256
16
256
1
1
256
16
256
128
128
256
1
1
256
128
128
128
16
16
1
1
16
16
128
128
16
16
1
1
16
16
128
1
16
16
1
1
16
16
1
128
128
128
1
1
128
128
128
128
256
128
1
1
128
256
128
128
16
16
1
1
16
16
128
128
16
16
1
1
16
16
128
128
16
16
1
1
16
16
128
128
16
16
1
1
16
16
128
8
16
16
1
1
16
16
8
256
256
16
1
1
16
256
256
256
256
16
1
1
16
256
256


In [39]:
n_max = onp.max([onp.max(tl.level_log2_size) for tl in topolevels])
print(n_max)

11


In [44]:
buffer_sweep_dummy = jnp.zeros((domain.n_elements + 1, element_type.n_basis))


Array([[ 6.35763037e-01,  6.35762661e-01,  5.97072472e-01, ...,  6.35762982e-01,  6.16896369e-01,  6.35762931e-01],
       [ 6.35761141e-01,  5.97062369e-01,  6.35761014e-01, ...,  6.16896906e-01,  6.35761142e-01,  6.16896916e-01],
       [ 6.35763035e-01,  6.35762984e-01,  5.97074546e-01, ...,  6.35763060e-01,  6.16896542e-01,  6.35762978e-01],
       ...,
       [ 8.84671112e-02, -1.22195832e-02,  6.09439717e-02, ...,  4.88757706e-02,  9.94858434e-02,  3.95570258e-02],
       [ 3.54395791e-06, -1.38177590e-02,  5.79427214e-02, ...,  3.54395791e-06,  3.53429284e-02, -1.36632277e-03],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00]], dtype=float64)

In [57]:
for i in range(len(topolevels)):
    %timeit jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[i], source_mapped, buffer_sweep_dummy, n_max).block_until_ready()

20.4 ms ± 122 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
69.1 ms ± 633 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
21.5 ms ± 65.7 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
73.8 ms ± 519 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
73.1 ms ± 477 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
20.8 ms ± 407 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
68.7 ms ± 230 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
21.2 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
42.6 ms ± 164 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
68.9 ms ± 120 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
68.5 ms ± 126 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


KeyboardInterrupt: 

In [58]:
print([topolevels[i].sizes for i in range(len(topolevels))])

[Array([2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048,
       2048, 2048, 1024, 1024, 1024,  512,  256], dtype=int32), Array([ 16,  16,  32,  32,  32,  32,  64,  64,  64,  64,  64,  64, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256,
       256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256

In [56]:
[jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[i], source_mapped, buffer_sweep_dummy, n_max).block_until_ready() for i in range(len(topolevels))]

[Array([[ 6.35763037e-01,  6.35762661e-01,  5.97072472e-01, ...,  6.35762982e-01,  6.16896369e-01,  6.35762931e-01],
        [ 6.35761141e-01,  5.97062369e-01,  6.35761014e-01, ...,  6.16896906e-01,  6.35761142e-01,  6.16896916e-01],
        [ 6.35763035e-01,  6.35762984e-01,  5.97074546e-01, ...,  6.35763060e-01,  6.16896542e-01,  6.35762978e-01],
        ...,
        [ 8.84671112e-02, -1.22195832e-02,  6.09439717e-02, ...,  4.88757706e-02,  9.94858434e-02,  3.95570258e-02],
        [ 3.54395791e-06, -1.38177590e-02,  5.79427214e-02, ...,  3.54395791e-06,  3.53429284e-02, -1.36632277e-03],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00]], dtype=float64),
 Array([[ 4.65055931e-06,  7.06407485e-06,  9.60434884e-02, ...,  4.51314832e-06,  4.92459872e-02,  4.28984478e-06],
        [ 1.62330478e-05,  9.60568585e-02,  2.25397655e-05, ...,  4.92415909e-02,  1.78783852e-05,  4.92427683e-02],
        [ 5.70455955e-06,  5.21076

In [46]:
jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[0], source_mapped, buffer_sweep_dummy, n_max).block_until_ready()

21.3 ms ± 402 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:

from jax_sn.solution_domain.transport_operator import _collapse_dij, _collapse_dij_vmap, _assert_shape_base

def _single_level_from_setup(geometric_domain : GeometricDomain,

In [25]:
%timeit transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy, int(onp.max(topolevel.level_log2_size))).block_until_ready()

20.8 ms ± 268 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [19]:
%timeit _setup_elements_idx_in_precomp_with_omega_n_calculation(topolevel.levels_argsort, *solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel.angle, source_mapped)[0].block_until_ready()

6.98 ms ± 69.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
%timeit  transport_inverse_sweep_complete_vmap_levels_with_callables(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy).block_until_ready()

20.3 ms ± 194 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
collapse_new = collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

In [23]:
%timeit collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

624 ms ± 7.24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
onp.testing.assert_allclose(collapse_new, buffer_sweep_dummy[:-1])

In [85]:
from jax_sn.solution_domain.transport_operator import _sweep_elements
%timeit _sweep_elements(toposorts[0].levels_argsort, buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, solution_domain.domain, abs_omega_dot_n_Je).block_until_ready()

782 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [86]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f

In [13]:
from stell_blanket_functions import  _matfree_nte_from_sb, _test_domain_from_sb

In [14]:
def check_sb():
    domain = _test_domain_from_sb(3,2)
    import StellBlanket as SB
    quad_set = SB.Generic_Elements.TN_Quadrature_Set(4)
    _, source = _matfree_nte_from_sb(3, 2, 4, 0)
    simulation = SB.Generic_Elements.Create_RAM_Simulation(quad_set, SB.Generic_Elements.Energy_Set([0,1]), domain, 2, 0)
    print(simulation.geometric_domain.connectivity.shape[0])
    simulation.simulation.Sweep(simulation.energy_set[0], simulation.quadrature_set[0], source)
    %timeit simulation.simulation.Total_Sweep(simulation.energy_set[0], source)

check_sb()

60000
817 ms ± 20.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
